In [207]:
import pandas as pd
import numpy as np

In [208]:
# 주문 상세 데이터 로딩
oi = pd.read_csv('../data/order_items.csv')
oi.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  500000 non-null  int64  
 1   order_id       500000 non-null  int64  
 2   product_id     500000 non-null  int64  
 3   quantity       500000 non-null  int64  
 4   unit_price     484887 non-null  float64
 5   discount       500000 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 22.9 MB


In [209]:
# oi.info 를 찍어서 column 들중에 null값이 있는걸 확인하고 데이터를 정제
oi.info()
#quantity -2
#unit price NaN 처리

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  500000 non-null  int64  
 1   order_id       500000 non-null  int64  
 2   product_id     500000 non-null  int64  
 3   quantity       500000 non-null  int64  
 4   unit_price     484887 non-null  float64
 5   discount       500000 non-null  float64
dtypes: float64(2), int64(4)
memory usage: 22.9 MB


In [210]:
#데이터 정제 : unit_price NaN 처리, quanity 음수 처리

# 데이터 정제 - unit_price NaN 처리, quantity 음수 처리

# 결측치 개수를 직접 확인할때
oi.isna().sum()
oi['unit_price'].isna().sum()

#데이터 타입 확인
oi['unit_price'].dtypes

oi = oi.dropna(subset=['unit_price'])
oi = oi[oi['quantity']>0]
# oi.info()

# oi = oi.dropna(subset=['unit_price'])   #unit_price 컬럼 기준으로 NaN들어있는 row를 삭제
# oi = oi[oi['quantity']>0] #0보다 큰거

In [211]:
#주문 금액(단가 * 수량 * (1 - 할인율)을 컬럼 생성

#주문금액 컬럼 생성  (amount)
#주문 금액 공식 :  unit price * quantity (1- discount rate)  

#1) assign을 사용한 방식(원본보호)
#assign은 새로운 DataFrame 을 반환하므로 결과를 다시 oi에 저장
# oi2  = oi.assign(
#     amount = oi['unit_price'] * oi['quantity'] * (1 - oi['discount'])
# )      

# oi2.describe()


#assign을 사용한 방식  (원본 보호)  기존 데이터 프레임에 column 만 추가
# oi2 = oi.assign(
#     amount = oi['unit_price'] * oi['quantity'] * (1- oi['discount']
# )

#vector연산을 사용한 방식 (메모리 절역) . 기존 oi 에 추가하는 형태
#vector 연산은 원소별 연산

oi['amount'] = oi['unit_price'] * oi['quantity'] * (1- oi['discount'])
oi.describe()
oi.info()

<class 'pandas.DataFrame'>
Index: 484400 entries, 0 to 499999
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  484400 non-null  int64  
 1   order_id       484400 non-null  int64  
 2   product_id     484400 non-null  int64  
 3   quantity       484400 non-null  int64  
 4   unit_price     484400 non-null  float64
 5   discount       484400 non-null  float64
 6   amount         484400 non-null  float64
dtypes: float64(3), int64(4)
memory usage: 29.6 MB


In [212]:
# 주문금액이 100000 이상이면 고액(true), 아니면(false) 일반을 저장하는 컬럼 생성
# 신규 column: grade
#np.where(조건식자리,  True일때 값, False일때 값)
# oi['grade'] = np.where(oi['amount']>100000, '고액', '일반')

# np.where 로 true/false 조건식으로 바로 column 만들어버리기   고액과 일반을 100000으로 구분해서
# 주문금액이 100000 이상이면 고액(true), 아니면(false) 일반을 저장하는 column 선택
# 신규 column 'grade'만들기

oi['grade'] = np.where(oi['amount']>100000, '고액', '일반')
# oi.head(10)

#방금 만든 신규 column grade가 만들어졌는지 확인하려면?
oi.head()

#grade가 뭐가 있는지 확인하고 각각 몇개가 있는지 보려면?
oi.grade.value_counts()

grade
일반    320448
고액    163952
Name: count, dtype: int64

In [213]:
oi.head()
#grade를 확인해보려면
oi.grade.value_counts()

grade
일반    320448
고액    163952
Name: count, dtype: int64

In [214]:
#다중 분류 : np.select(조건들, 값들, 기본값)
#주문 금액 > 200000 :최고액, 주문금액 >= 50000이면 고액 , 나머지: 일반

#conditionlist
#choicelist
#oi['tier'] = np.select(conditionlist, choicelist, default = '일반')

conditionlist=[oi['amount']>=200000, oi['amount']>= 50000]
choicelist = ["최고액", "고액"]

#np.select()로 만든 분류 결과를 새로운  column 'tier'에 저장하기

oi['tier'] = np.select(conditionlist, choicelist, default='일반' )

#value_counts()
#새로 만든 column의 .value_counts()로 새로 추가된 분류와 그 값들을 찾을 수 있다
oi['tier'].value_counts()
# tier
# 일반     214435
# 고액     182057
# 최고액     87908


tier
일반     214435
고액     182057
최고액     87908
Name: count, dtype: int64

In [215]:
type(oi['tier']) #Series
type(oi) #DataFrame

pandas.DataFrame

In [216]:
#새로운 dictionary 생성하기 
data = {
    'name': ['홍길동', '고길동', '마이꼴'],
    'kor' : [100, 90, 90],
    'eng' : [50, 90, 100]
}

type(data)

dict

In [217]:
#Data라는 dictionary를 DataFrame 로 변환한것을 students에 다가 넣음
students = pd.DataFrame(data)

In [218]:
type(students) #DataFrame

pandas.DataFrame

In [219]:
#set_index() 를 사용해서 특정 column을 index로 만들기
#여기서 'name'이 index로 사용될 column임
#name을 index로 사용하면 좋은 점은? 이제 loc를 사용하여 name으로 바로 조회할 수 있음
#eg. students.loc['홍길동']

#inplace = True 는 새로운 DataFrame 을 사용하는 대신 기존 students 자체를 직접 변경한다

students.set_index('name', inplace=True)

In [220]:
students.loc['홍길동', 'kor']

np.int64(100)

In [226]:
#apply
#lambda 와 axis=1 은 apply()가각 학생의 한 row 씩 계산하도록 만드는 부분
students['tot'] = students.apply(
    lambda row: row['kor'] + row['eng'],
    axis = 1
)
#lambda는 간단한 함수를 한줄로 만드는 문법
# lambda 입력값: 반환할_계산식

#tot라는 새로운 column을 만들고
#기존의 3개의 rows( 홍길동, 고길동, 마이꼴)에 값을 새로운 tot 값을 하나씩 넣음
#          kor  eng  tot
# name
# 홍길동    100   50  150
# 고길동     90   90  180
# 마이콜     90  100  190


students['tot']
students[['kor', 'eng']]


,kor,eng
name,,
홍길동,100,50
고길동,90,90
마이꼴,90,100


In [ ]:
# axis = 1 은 각 행을 가로로 더하라는 뜻
# 각 학생의 같은 행에 있는 kor 값과 eng 값을 가로로 더하라는 뜻이야.
students['tot'] = students[['kor', 'eng']].sum(axis=1)



In [ ]:
#axis 는 2가지  ( 0 or  1 )
# axis 0은 vertical 방향

In [ ]:
students.sum(axis=0)

kor    280
eng    240
dtype: int64

In [ ]:
students.sum(axis =1)

name
홍길동    150
고길동    180
마이꼴    190
dtype: int64

In [ ]:
# apply + lambda
# students['tot'] = students.apply(lambda row: row['kor']+row['eng'], axis=1)

# apply, sum 함수 
# students['tot'] = students[['kor', 'eng']].apply(sum, axis=1)

#vector 함수
# 병렬 처리 가능 (한번에 처리 - 3개의 row가 한번에 만들어짐)
#students['tot'] = students['kor'] +students['eng']

# sum()
# students['tot'] = students[['kor', 'eng']].sum(axis=1)


,kor,eng,tot
name,,,
홍길동,100,50,150
고길동,90,90,180
마이꼴,90,100,190


In [ ]:
#np.where 는 조건이 많아서 복잡해지면 코드가 지져분해진다
#금액과 할인율을 함께 보는 사용자 정의 라벨

#apply 는 dataframe 에 적용됨
#주문금액이 10만원 이상이고 할인율이 0.3 이상일때 
#'고액 - 대폭할인' 이라는 라벨을 주고 아니면 '기타'
# apply method 사용해서
#apply(function, axis)

def label(row):
    if row['amount'] >= 100000 and row['discount'] >= 0.4:
        return '고액-대폭할인'
    return '기타'

In [97]:
oi.apply(label, axis=1).value_counts()

기타         457328
고액-대폭할인     27072
Name: count, dtype: int64

In [95]:
condtion1= oi['amount']>=100000
condition2 = oi['discount']>=0.3
np.where(condtion1&condition2, '고액-대폭할인', '기타')

array(['기타', '기타', '기타', ..., '기타', '기타', '기타'],
      shape=(484400,), dtype='<U7')

In [79]:
oi.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount,amount,grade,tier
0,59256,114625,3,1,20300.0,0.05,19285.0,일반,일반
1,230587,66337,80,2,90000.0,0.45,99000.0,일반,고액
2,279813,163343,3,3,20300.0,0.23,46893.0,일반,일반
3,88487,180332,79,1,8600.0,0.38,5332.0,일반,일반
4,39240,174179,232,1,42600.0,0.31,29394.0,일반,일반
